# Notebook 4 — Recovery trajectories and candidate families

Twelve fixed study areas, two spatial supports, one sequence:
**spatial context → observed trajectories → analysis matrix → preprocessing → k-means diagnostics → recovery families → maps.**

The main signal is reliability-qualified NTL (`DNB_BRDF_Corrected_NTL`, MQF == 0). Municipality-wide means the GHSL G3 support within the whole boundary. Each local support is the same municipality's G3 pixels inside a 5×5 VIIRS window, anchored on its brightest pre-event baseline pixel. It can contain fewer than 25 eligible pixels. These are systematic anchors, not the named POIs in Notebook 3.

Run sequentially in the existing Black Marble environment. This revision is supplied with cleared outputs: the source datasets could not be accessed here, so no new empirical results are claimed. Existing project and export locations are retained. The two boundary-column settings below need confirmation against the local shapefile; they are explicit settings rather than a name-resolution framework.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray as rxr
from rasterio.enums import Resampling
from rasterio.features import rasterize
from shapely.geometry import box, Point
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from IPython.display import display


In [ ]:
PROJECT_DIR_OVERRIDE = Path(
    "/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/"
    "02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery"
)
PROJECT_DIR = PROJECT_DIR_OVERRIDE
DATA_DIR = PROJECT_DIR / "datasets"
VNP46_DIR = DATA_DIR / "VNP46"
PROCESSED_DIR = VNP46_DIR / "processed"
A2_ZARR_PATH = PROCESSED_DIR / "Haiyan_VNP46A2.zarr"
MUNICIPALITIES_PATH = DATA_DIR / "boundaries/MuniCities/MuniCities.shp"
HAIYAN_TRACK_PATH = DATA_DIR / "yolanda-path-line-/Yolanda Path Line.shp"
# Fixed filename; its directory was not recorded in the source notebook.
ghsl_paths = list(DATA_DIR.glob("**/GHSL_SMOD_E2015.tif"))
assert len(ghsl_paths) == 1, "Set GHSL_PATH to the existing GHSL_SMOD_E2015.tif."
GHSL_PATH = ghsl_paths[0]
NAME_COLUMN, PROVINCE_COLUMN = "NAME_2", "PROVINCE"

OUTPUT_DIR = PROJECT_DIR / "output/focused_recovery_walkthrough"
FIGURE_DIR, TABLE_DIR = OUTPUT_DIR / "figures", OUTPUT_DIR / "tables"
STUDY_AREAS = ["Tacloban", "Ormoc", "Baybay", "Catbalogan", "Borongan", "Guiuan",
               "Palo", "Tanauan", "Tolosa", "Dulag", "Basey", "Lawaan"]
study_area_provinces = dict(zip(STUDY_AREAS, [
    "Leyte", "Leyte", "Leyte", "Samar", "Eastern Samar", "Eastern Samar",
    "Leyte", "Leyte", "Leyte", "Leyte", "Samar", "Eastern Samar"
]))
EVENT_DATE = pd.Timestamp("2013-11-08")
BASELINE_DAYS, AGGREGATION_DAYS, KERNEL_SIZE = 60, 4, 5
BASELINE_START = EVENT_DATE - pd.Timedelta(days=BASELINE_DAYS)
PRE_EVENT_END = EVENT_DATE - pd.Timedelta(days=1)
ANALYSIS_START = EVENT_DATE - pd.Timedelta(days=180)
PROFILE_END = EVENT_DATE + pd.Timedelta(days=363)
DNB_BAND, MQF_BAND = "DNB_BRDF_Corrected_NTL", "Mandatory_Quality_Flag"
SPATIAL_DIMS = ("y", "x")
GHSL_MASKS, SETTLEMENT_MASK = {"G3": (22, 23, 30)}, "G3"
SPATIAL_COMPLETENESS_PCT = 10.0  # Original walkthrough setting; not an RQ1 optimum.
RQ_CLIP_PERCENTILE = 95.0
MIN_BASELINE_OBSERVATIONS = {4: 3}
CLUSTER_HORIZON_DAYS, CLUSTER_BIN_DAYS, MIN_BIN_COMPOSITES = 180, 20, 2
CLUSTER_RANDOM_STATE = 42
SUPPORTS = ["Municipality-wide G3", "Local 5×5 Kernel"]


**Retained processing choices.** Four-day medians, a 60-day pixelwise baseline, at least three baseline composites per fixed pixel, and daily spatial P95 capping are unchanged. The original **10% spatial-completeness gate is permissive** and is retained for continuity, not presented as the RQ1 optimum. Family labels are conditional on this setting; review the coverage plots before interpreting them. Missing bins remain missing.


In [ ]:
PLOT_FONT, PLOT_TEXT_COLOR = "Arial", "#243B5A"
EVENT_COLOR, BASELINE_COLOR = "#0057FF", "#6C7882"
AREA_COLORS = dict(zip(STUDY_AREAS, px.colors.qualitative.Dark24))
FAMILY_COLORS = ["#0072B2", "#E69F00", "#D55E00", "#009E73", "#CC79A7", "#56B4E9"]

def style_figure(fig, title):
    fig.update_layout(
        template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="white",
        width=1280, height=720, font=dict(family=PLOT_FONT, size=14, color=PLOT_TEXT_COLOR),
        title=dict(text=title, x=0.03, xanchor="left", font=dict(size=22)),
        margin=dict(l=75, r=45, t=90, b=115),
        legend=dict(orientation="h", x=0, y=-0.18, xanchor="left", yanchor="top"),
    )
    fig.update_xaxes(zeroline=False, gridcolor="#E8EDF3")
    fig.update_yaxes(zeroline=False, gridcolor="#E8EDF3")
    return fig

def finish_figure(fig, filename):
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    fig.write_html(FIGURE_DIR / f"{filename}.html", include_plotlyjs="directory")
    fig.show()

def boundary_coordinates(geometry):
    boundary = geometry.boundary
    lines = list(boundary.geoms) if boundary.geom_type == "MultiLineString" else [boundary]
    x, y = [], []
    for line in lines:
        xx, yy = line.xy
        x.extend([*xx, None]); y.extend([*yy, None])
    return x, y

def add_haiyan_marker(fig, row=None, col=None):
    fig.add_vline(x=EVENT_DATE, line_dash="dash", line_color=EVENT_COLOR,
                  line_width=1.5, row=row, col=col)


## 1. Fix the spatial supports

Study-area and province matching is explicit. The short removal of “City of”/“City” below only translates boundary labels to the existing study names. It does not search for alternative columns or choose among ambiguous matches. A failed match stops here, before any recovery is calculated.


In [ ]:
municipalities = gpd.read_file(MUNICIPALITIES_PATH)
assert {NAME_COLUMN, PROVINCE_COLUMN}.issubset(municipalities.columns), (
    f"Set NAME_COLUMN and PROVINCE_COLUMN above. Available fields: {list(municipalities.columns)}"
)
municipalities["unit_name"] = (municipalities[NAME_COLUMN].str.upper()
    .str.replace("CITY OF ", "", regex=False).str.replace(" CITY", "", regex=False).str.strip().str.title())
municipalities["province_name"] = municipalities[PROVINCE_COLUMN].str.strip().str.title()
focused_municipalities = municipalities.loc[
    municipalities.unit_name.isin(STUDY_AREAS)
    & municipalities.province_name.eq(municipalities.unit_name.map(study_area_provinces))
].copy()
assert focused_municipalities.unit_name.value_counts().reindex(STUDY_AREAS, fill_value=0).eq(1).all(), (
    "Expected one boundary per study area. Check the explicit names and province labels."
)
focused_municipalities = focused_municipalities.set_index("unit_name").loc[STUDY_AREAS].reset_index()
focused_municipalities["profile_id"] = np.arange(1, len(STUDY_AREAS) + 1)
municipalities_display = focused_municipalities.to_crs("EPSG:4326")
regional_boundaries = municipalities.to_crs("EPSG:4326")
haiyan_track = gpd.read_file(HAIYAN_TRACK_PATH).to_crs("EPSG:4326")

# The saved source run identifies this VIIRS grid as EPSG:4326.
a2 = xr.open_zarr(A2_ZARR_PATH, consolidated=None, chunks="auto", mask_and_scale=True)
a2 = a2.set_coords("date").swap_dims({a2.date.dims[0]: "date"}).sortby("date")
a2 = a2.assign_coords(date=pd.to_datetime(a2.date.values).normalize())
assert a2.indexes["date"].is_unique, "Resolve duplicate observation dates before compositing."
a2 = a2.sel(date=slice(ANALYSIS_START, PROFILE_END)).rio.write_crs("EPSG:4326")
dnb = a2[DNB_BAND].astype("float32")
dnb = dnb.where(np.isfinite(dnb) & (dnb >= 0) & ~dnb.isin([6553.5, 65535.0]))
mqf = a2[MQF_BAND]
ghsl = rxr.open_rasterio(GHSL_PATH, masked=True).isel(band=0, drop=True)
ghsl_viirs = ghsl.rio.reproject_match(dnb.isel(date=0), resampling=Resampling.nearest)
ghsl_viirs = ghsl_viirs.assign_coords(x=dnb.x, y=dnb.y)
ghsl_mask = ghsl_viirs.isin(GHSL_MASKS[SETTLEMENT_MASK])
municipalities_raster_crs = focused_municipalities.to_crs(a2.rio.crs)
all_zone_id = xr.DataArray(rasterize(
    list(zip(municipalities_raster_crs.geometry, municipalities_raster_crs.profile_id)),
    out_shape=(dnb.sizes["y"], dnb.sizes["x"]), transform=dnb.rio.transform(),
    fill=0, all_touched=False, dtype="int32"
), dims=SPATIAL_DIMS, coords={"y": dnb.y, "x": dnb.x})
rq_base_mask = ghsl_mask & (all_zone_id > 0)
rq_unclipped = dnb.where((mqf == 0) & rq_base_mask)
rq_daily_p95 = rq_unclipped.chunk({"y": -1, "x": -1}).quantile(
    RQ_CLIP_PERCENTILE / 100, dim=SPATIAL_DIMS, skipna=True).compute()
rq_cube = xr.where(rq_unclipped > rq_daily_p95, rq_daily_p95, rq_unclipped)


In [ ]:
def build_pixel_matched_profile(cube, support_mask, unit_name, unit_type):
    # Reindex every calendar block: wholly absent periods must remain visible.
    blocks = np.arange((ANALYSIS_START - EVENT_DATE).days // AGGREGATION_DAYS,
                       (PROFILE_END - EVENT_DATE).days // AGGREGATION_DAYS + 1)
    day_blocks = (pd.DatetimeIndex(cube.date.values) - EVENT_DATE).days // AGGREGATION_DAYS
    composites = (cube.where(support_mask).assign_coords(block=("date", day_blocks))
        .groupby("block").median("date", skipna=True).reindex(block=blocks).compute())
    baseline_blocks = composites.sel(block=slice(-BASELINE_DAYS // AGGREGATION_DAYS, -1))
    ntl0 = baseline_blocks.median("block", skipna=True)
    fixed_mask = (support_mask & (baseline_blocks.count("block") >= MIN_BASELINE_OBSERVATIONS[4])
                  & np.isfinite(ntl0) & (ntl0 > 0))
    fixed_pixel_count = int(fixed_mask.sum())
    paired_valid = composites.notnull() & fixed_mask
    valid_pixel_count = paired_valid.sum(SPATIAL_DIMS)
    spatial_coverage_pct = 100 * valid_pixel_count / max(fixed_pixel_count, 1)
    current_radiance = composites.where(paired_valid).sum(SPATIAL_DIMS, min_count=1)
    matched_baseline = ntl0.where(paired_valid).sum(SPATIAL_DIMS, min_count=1)
    reduced = xr.Dataset({
        "recovery_pct": 100 * current_radiance / matched_baseline,
        "current_radiance": current_radiance,
        "matched_baseline_radiance": matched_baseline,
        "raw_median_ntl": composites.where(paired_valid).median(SPATIAL_DIMS, skipna=True),
        "spatial_coverage_pct": spatial_coverage_pct,
        "valid_pixel_count": valid_pixel_count,
    })
    profile = reduced.to_dataframe().reset_index()
    profile["date_start"] = EVENT_DATE + pd.to_timedelta(profile.block * AGGREGATION_DAYS, unit="D")
    profile["date_end"] = profile.date_start + pd.Timedelta(days=AGGREGATION_DAYS - 1)
    profile["date_mid"] = profile.date_start + pd.Timedelta(days=1.5)
    withheld = profile.spatial_coverage_pct.lt(SPATIAL_COMPLETENESS_PCT)
    profile.loc[withheld, ["recovery_pct", "current_radiance", "matched_baseline_radiance", "raw_median_ntl"]] = np.nan
    profile["observation_status"] = np.where(profile.recovery_pct.notna(), "observed", "not observable")
    profile["unit_name"], profile["unit_type"] = unit_name, unit_type
    profile["fixed_baseline_pixels"] = fixed_pixel_count
    profile["ghsl_mask"], profile["sc_threshold_pct"] = SETTLEMENT_MASK, SPATIAL_COMPLETENESS_PCT
    profile["baseline_start"], profile["baseline_end"] = BASELINE_START, PRE_EVENT_END
    profile["aggregation_days"] = AGGREGATION_DAYS
    baseline_values = ntl0.where(fixed_mask).values
    report = dict(unit_name=unit_name, unit_type=unit_type, fixed_baseline_pixels=fixed_pixel_count,
                  baseline_median_ntl=float(np.nanmedian(baseline_values)) if fixed_pixel_count else np.nan)
    return profile, composites.where(fixed_mask), ntl0.where(fixed_mask), fixed_mask, report

municipality_profiles, kernel_profiles = [], []
municipality_reports, kernel_reports, kernel_anchors = [], [], []
baseline_surfaces, fixed_masks, municipality_composites = {}, {}, {}
kernel_windows = {}
for row in municipalities_raster_crs.itertuples():
    support = (all_zone_id == row.profile_id) & ghsl_mask
    yy, xx = np.where(support.values)
    assert len(yy), f"{row.unit_name}: no G3 support on the VIIRS grid."
    sy, sx = slice(yy.min(), yy.max()+1), slice(xx.min(), xx.max()+1)
    profile, composites, baseline, fixed, report = build_pixel_matched_profile(
        rq_cube.isel(y=sy, x=sx), support.isel(y=sy, x=sx), row.unit_name, SUPPORTS[0])
    profile["profile_id"] = row.profile_id
    municipality_profiles.append(profile); municipality_reports.append(report)
    baseline_surfaces[row.profile_id], fixed_masks[row.profile_id] = baseline, fixed
    municipality_composites[row.unit_name] = composites
    if not report["fixed_baseline_pixels"]:
        continue  # This area remains in the eligibility table as unavailable.
    iy, ix = np.unravel_index(np.nanargmax(baseline.values), baseline.shape)
    anchor_x, anchor_y = float(baseline.x[ix]), float(baseline.y[iy])
    gx, gy = int(np.abs(dnb.x.values-anchor_x).argmin()), int(np.abs(dnb.y.values-anchor_y).argmin())
    assert 2 <= gx < dnb.sizes["x"]-2 and 2 <= gy < dnb.sizes["y"]-2, "Anchor too close to raster edge."
    sy, sx = slice(gy-2, gy+3), slice(gx-2, gx+3)
    profile, _, _, _, report = build_pixel_matched_profile(
        rq_cube.isel(y=sy, x=sx), support.isel(y=sy, x=sx), row.unit_name, SUPPORTS[1])
    profile["profile_id"] = row.profile_id
    kernel_profiles.append(profile); kernel_reports.append(report)
    dx, dy = abs(float(dnb.x[1]-dnb.x[0])), abs(float(dnb.y[1]-dnb.y[0]))
    kernel_windows[row.unit_name] = box(anchor_x-2.5*dx, anchor_y-2.5*dy,
                                         anchor_x+2.5*dx, anchor_y+2.5*dy)
    kernel_anchors.append(dict(profile_id=row.profile_id, unit_name=row.unit_name,
        anchor_x=anchor_x, anchor_y=anchor_y, anchor_baseline_ntl=float(baseline.values[iy, ix]),
        g3_municipal_pixels_in_window=int(support.isel(y=sy, x=sx).sum())))
municipality_four_day = pd.concat(municipality_profiles, ignore_index=True)
kernel_four_day = (pd.concat(kernel_profiles, ignore_index=True) if kernel_profiles
                   else municipality_four_day.iloc[:0].copy())
municipality_reports = pd.DataFrame(municipality_reports)
kernel_reports = pd.DataFrame(kernel_reports, columns=municipality_reports.columns)
kernel_anchors = pd.DataFrame(kernel_anchors, columns=["profile_id", "unit_name", "anchor_x", "anchor_y",
                                                     "anchor_baseline_ntl", "g3_municipal_pixels_in_window"])


In [ ]:
# The same extent and contextual layers are reused for the family maps.
x0, y0, x1, y1 = municipalities_display.total_bounds
map_bounds = [x0-0.12, y0-0.12, x1+0.12, y1+0.12]
regional_display = regional_boundaries.cx[map_bounds[0]:map_bounds[2], map_bounds[1]:map_bounds[3]]
context_x, context_y = [], []
for geometry in regional_display.geometry:
    xx, yy = boundary_coordinates(geometry)
    context_x.extend(xx); context_y.extend(yy)
track_x, track_y = [], []
for geometry in haiyan_track.geometry:
    lines = list(geometry.geoms) if geometry.geom_type == "MultiLineString" else [geometry]
    for line in lines:
        xx, yy = line.xy
        track_x.extend([*xx, None]); track_y.extend([*yy, None])

def map_context(fig, row, col):
    fig.add_trace(go.Scatter(x=context_x, y=context_y, mode="lines", line=dict(color="#CFD6DE", width=0.7),
                            showlegend=False, hoverinfo="skip"), row=row, col=col)
    for color, width in [("white", 5), ("#111111", 2)]:
        fig.add_trace(go.Scatter(x=track_x, y=track_y, mode="lines", line=dict(color=color, width=width),
                                showlegend=False, hoverinfo="skip"), row=row, col=col)
    fig.update_xaxes(range=[map_bounds[0], map_bounds[2]], ticksuffix="°E", title_text="Longitude", row=row, col=col)
    fig.update_yaxes(range=[map_bounds[1], map_bounds[3]], ticksuffix="°N", title_text="Latitude", row=row, col=col)
    axis_number = (row-1)*2+col
    fig.update_yaxes(scaleanchor="x" if axis_number == 1 else f"x{axis_number}",
                     scaleratio=1/np.cos(np.deg2rad((y0+y1)/2)), row=row, col=col)

fig = make_subplots(rows=1, cols=2, subplot_titles=["Municipalities", "5×5 Kernels"])
for col in (1, 2):
    map_context(fig, 1, col)
    for row in municipalities_display.itertuples():
        color = AREA_COLORS[row.unit_name]
        xx, yy = boundary_coordinates(row.geometry)
        fig.add_trace(go.Scatter(x=xx, y=yy, mode="lines", line=dict(color=color, width=2 if col==1 else 0.8),
                                showlegend=False, hoverinfo="skip"), row=1, col=col)
        point = row.geometry.representative_point()
        if col == 2 and row.unit_name in kernel_windows:
            xx, yy = boundary_coordinates(kernel_windows[row.unit_name])
            fig.add_trace(go.Scatter(x=xx, y=yy, mode="lines", line=dict(color=color, width=2),
                                    showlegend=False, hoverinfo="skip"), row=1, col=col)
            anchor = kernel_anchors.set_index("unit_name").loc[row.unit_name]
            point = Point(anchor.anchor_x, anchor.anchor_y)
        fig.add_trace(go.Scatter(x=[point.x], y=[point.y], mode="markers+text", text=[row.unit_name],
            textposition="top center", textfont=dict(size=10), marker=dict(color=color, size=6),
            showlegend=False, hovertemplate=row.unit_name+"<extra></extra>"), row=1, col=col)
style_figure(fig, "")
finish_figure(fig, "01_study_areas_and_5x5")


support_counts = pd.concat([municipality_reports, kernel_reports], ignore_index=True)
fig = px.bar(support_counts, x="unit_name", y="fixed_baseline_pixels", color="unit_type", barmode="group",
             color_discrete_sequence=["#174A7E", "#E69F00"],
             labels={"fixed_baseline_pixels":"Fixed baseline-lit G3 pixels", "unit_name":"Study area", "unit_type":"Support"})
fig.update_yaxes(type="log")
style_figure(fig, "")
finish_figure(fig, "02_fixed_support_counts")


## 2. Observe the trajectories before summarising them

The paired support plots use the same baseline definition and completeness gate. Lines stop at missing four-day blocks. Coverage strips show the fraction of the fixed baseline-lit G3 support observed in each composite; they measure observability, not recovery. The horizontal line is 100% of the matched baseline. The vertical dashed blue line marks Haiyan on 8 November 2013.


In [ ]:
for support_name, profiles, slug in [(SUPPORTS[0], municipality_four_day, "municipality"),
                                     (SUPPORTS[1], kernel_four_day, "5x5")]:
    fig = make_subplots(rows=3, cols=4, subplot_titles=STUDY_AREAS,
                        shared_xaxes=True, vertical_spacing=0.10)
    for i, name in enumerate(STUDY_AREAS):
        r, c = i%3+1, i//3+1
        group = profiles.loc[profiles.unit_name.eq(name)]
        fig.add_trace(go.Scatter(x=group.date_mid, y=group.recovery_pct, mode="lines+markers",
            connectgaps=False, line=dict(color=AREA_COLORS[name], width=1.5), marker=dict(size=3),
            customdata=group[["spatial_coverage_pct", "valid_pixel_count"]], showlegend=False,
            hovertemplate="%{x|%d %b %Y}<br>%{y:.1f}% baseline<br>Coverage %{customdata[0]:.1f}%"
                          "<br>Valid pixels %{customdata[1]}<extra></extra>"), row=r, col=c)
        add_haiyan_marker(fig, r, c)
        fig.add_hline(y=100, line_color=BASELINE_COLOR, line_dash="dot", row=r, col=c)
        fig.update_xaxes(range=[BASELINE_START, PROFILE_END], tickformat="%b %Y", row=r, col=c)
        fig.update_yaxes(title_text="% baseline" if c==1 else None, row=r, col=c)
    style_figure(fig, f"{support_name} | Observed four-day recovery trajectories")
    fig.update_layout(height=700)

    finish_figure(fig, f"03_{slug}_trajectories")
    coverage = profiles.pivot(index="unit_name", columns="date_mid", values="spatial_coverage_pct").reindex(STUDY_AREAS)
    fig = go.Figure(go.Heatmap(x=coverage.columns, y=coverage.index, z=coverage.values,
        zmin=0, zmax=100, colorscale="Greens", colorbar=dict(title="SC %"),
        hovertemplate="%{y}<br>%{x|%d %b %Y}<br>Observed fixed pixels %{z:.1f}%<extra></extra>"))
    add_haiyan_marker(fig)
    style_figure(fig, f"{support_name} | Observability behind the trajectories")
    fig.update_xaxes(range=[BASELINE_START, PROFILE_END])
    fig.update_layout(height=500)
    finish_figure(fig, f"04_{slug}_coverage")


In [ ]:
# A compact check of absolute radiance and the influence of baseline brightness.
brightness_rows = []
for support_name, profiles, reports in [(SUPPORTS[0], municipality_four_day, municipality_reports),
                                        (SUPPORTS[1], kernel_four_day, kernel_reports)]:
    for name, group in profiles.groupby("unit_name", sort=False):
        pre = group.loc[group.date_start.between(BASELINE_START, PRE_EVENT_END), "recovery_pct"].dropna()
        report = reports.set_index("unit_name").loc[name]
        brightness_rows.append(dict(unit_name=name, unit_type=support_name,
            baseline_median_ntl=report.baseline_median_ntl,
            fixed_baseline_pixels=report.fixed_baseline_pixels,
            pre_event_mad_pct=(pre-pre.median()).abs().median()))
brightness_diagnostic = pd.DataFrame(brightness_rows)
fig = px.scatter(brightness_diagnostic, x="baseline_median_ntl", y="pre_event_mad_pct",
    color="unit_type", symbol="unit_type", hover_name="unit_name",
    hover_data=["fixed_baseline_pixels"], color_discrete_sequence=["#174A7E", "#E69F00"],
    labels={"baseline_median_ntl":"Baseline radiance (nW cm⁻² sr⁻¹)",
            "pre_event_mad_pct":"Pre-event MAD (percentage points)", "unit_type":"Support"})
style_figure(fig, "Check dim or volatile baselines before clustering")
finish_figure(fig, "05_baseline_variability")


In [ ]:
# ------------------------------------------------------------
# Baseline median, spatial IQR, and temporal MAD
# ------------------------------------------------------------

brightness_rows = []
area_ids = municipalities_raster_crs.set_index("unit_name").profile_id

for support_name, profiles, reports in [
    (SUPPORTS[0], municipality_four_day, municipality_reports),
    (SUPPORTS[1], kernel_four_day, kernel_reports),
]:
    report_lookup = reports.set_index("unit_name")

    for name, group in profiles.groupby("unit_name", sort=False):
        pre = group.loc[
            group.date_start.ge(BASELINE_START)
            & group.date_end.le(PRE_EVENT_END)
        ]
        observed_pre = pre.recovery_pct.dropna()
        report = report_lookup.loc[name]

        q25 = median = q75 = np.nan
        profile_id = area_ids.loc[name]

        if profile_id in baseline_surfaces:
            surface = baseline_surfaces[profile_id]

            if support_name == SUPPORTS[1]:
                window = kernel_windows.get(name)

                if window is not None:
                    west, south, east, north = window.bounds
                    surface = surface.where(
                        (surface.x >= west) & (surface.x <= east)
                        & (surface.y >= south) & (surface.y <= north)
                    )
                else:
                    surface = surface.where(False)

            values = surface.values
            values = values[np.isfinite(values)]

            if values.size:
                q25, median, q75 = np.quantile(
                    values, [0.25, 0.50, 0.75]
                )

        brightness_rows.append({
            "unit_name": name,
            "unit_type": support_name,
            "baseline_median_ntl": median,
            "baseline_q25": q25,
            "baseline_q75": q75,
            "fixed_baseline_pixels": report.fixed_baseline_pixels,
            "pre_event_mad_pct": (
                observed_pre - observed_pre.median()
            ).abs().median(),
            "valid_pre_composites": len(observed_pre),
            "median_pre_coverage_pct": pre.spatial_coverage_pct.median(),
        })

brightness_diagnostic = pd.DataFrame(brightness_rows)

municipal_map_metrics = brightness_diagnostic.loc[
    brightness_diagnostic.unit_type.eq(SUPPORTS[0])
].copy()

municipal_map_metrics["province_name"] = (
    municipal_map_metrics.unit_name.map(study_area_provinces)
)

map_gdf = regional_boundaries.loc[
    regional_boundaries.province_name.isin([
        "Samar", "Eastern Samar", "Northern Samar",
        "Leyte", "Southern Leyte",
    ])
].copy()

map_gdf = map_gdf.merge(
    municipal_map_metrics,
    on=["unit_name", "province_name"],
    how="left",
    validate="many_to_one",
)

# Annotation offsets in screen pixels.
# Adjust individual entries here if labels overlap in your final export.
scatter_offsets = {
    "Tacloban":   (-65, -22),
    "Ormoc":     (-60,  24),
    "Baybay":    ( 55, -24),
    "Catbalogan":( 65,  20),
    "Borongan":  (-65, -36),
    "Guiuan":    (-65,  36),
    "Palo":      ( 55, -38),
    "Tanauan":   ( 65,  36),
    "Tolosa":    (-55, -52),
    "Dulag":     ( 55,  52),
    "Basey":     (-60,  52),
    "Lawaan":    ( 60, -52),
}

map_offsets = {
    "Tacloban":   (-70, -38),
    "Ormoc":      (-45, -20),
    "Baybay":     (-40,  30),
    "Catbalogan": (-50, -28),
    "Borongan":   ( 50, -25),
    "Guiuan":     ( 45,  28),
    "Palo":       (-80,  -4),
    "Tanauan":    (-85,  22),
    "Tolosa":     (-65,  48),
    "Dulag":      ( 48,  48),
    "Basey":      ( 60, -48),
    "Lawaan":     ( 65,   4),
}

fig = make_subplots(
    rows=1,
    cols=3,
    column_widths=[0.40, 0.30, 0.30],
    horizontal_spacing=0.055,
    subplot_titles=[
        "Median brightness versus variability",
        "Median pixel baseline radiance",
        "Pre-event trajectory MAD",
    ],
)


# ------------------------------------------------------------
# 1. Scatter without labels or IQR bars
# ------------------------------------------------------------

for support, colour, symbol, suffix in [
    (SUPPORTS[0], "#174A7E", "circle", "M"),
    (SUPPORTS[1], "#E69F00", "diamond", "K"),
]:
    subset = brightness_diagnostic.loc[
        brightness_diagnostic.unit_type.eq(support)
    ].dropna(subset=["baseline_median_ntl", "pre_event_mad_pct"])

    fig.add_trace(
        go.Scatter(
            x=subset.baseline_median_ntl,
            y=subset.pre_event_mad_pct,
            mode="markers",
            name=f"{suffix} · {support}",
            marker=dict(
                color=colour,
                symbol=symbol,
                size=14,
                line=dict(color="white", width=1),
            ),
            customdata=subset[[
                "unit_name",
                "fixed_baseline_pixels",
                "valid_pre_composites",
                "median_pre_coverage_pct",
            ]].to_numpy(),
            hovertemplate=(
                "<b>%{customdata[0]}</b>"
                "<br>Median baseline: %{x:.2f}"
                "<br>Temporal MAD: %{y:.2f} percentage points"
                "<br>Fixed pixels: %{customdata[1]}"
                "<br>Valid composites: %{customdata[2]}"
                "<br>Median coverage: %{customdata[3]:.1f}%"
                "<extra>%{fullData.name}</extra>"
            ),
        ),
        row=1, col=1,
    )

# ------------------------------------------------------------
# 2–3. Filled municipality maps
# ------------------------------------------------------------

west, south, east, north = map_gdf.total_bounds
map_aspect = 1 / np.cos(np.deg2rad((south + north) / 2))

for col, metric, scale, units, bar_x in [
    (2, "baseline_median_ntl", "Viridis", "nW cm⁻² sr⁻¹", 0.68),
    (3, "pre_event_mad_pct", "YlOrRd", "Percentage points", 1.01),
]:
    finite_values = map_gdf[metric].dropna()
    colour_max = (
        float(finite_values.max())
        if len(finite_values) and finite_values.max() > 0
        else 1.0
    )

    # Draw the regional context first; calculated municipalities on top.
    ordered = map_gdf.assign(
        has_value=map_gdf[metric].notna()
    ).sort_values("has_value")

    for _, row in ordered.iterrows():
        value = row[metric]
        colour = (
            px.colors.sample_colorscale(
                scale, [float(value) / colour_max]
            )[0]
            if pd.notna(value)
            else "#E5E9EE"
        )

        # Separate polygons avoid connecting multipart municipalities.
        polygons = (
            list(row.geometry.geoms)
            if row.geometry.geom_type == "MultiPolygon"
            else [row.geometry]
        )

        for polygon in polygons:
            xx, yy = boundary_coordinates(polygon)

            fig.add_trace(
                go.Scatter(
                    x=xx,
                    y=yy,
                    mode="lines",
                    fill="toself",
                    fillcolor=colour,
                    line=dict(
                        color="#64748B" if pd.notna(value) else "white",
                        width=0.8 if pd.notna(value) else 0.4,
                    ),
                    showlegend=False,
                    hovertemplate=(
                        f"<b>{row.unit_name}</b><br>"
                        + (
                            f"{value:.2f} {units}"
                            if pd.notna(value)
                            else "No calculated value"
                        )
                        + "<extra></extra>"
                    ),
                ),
                row=1, col=col,
            )

    # A colourbar tied to the same scale used for polygon fills.
    fig.add_trace(
        go.Scatter(
            x=[None, None],
            y=[None, None],
            mode="markers",
            marker=dict(
                color=[0, colour_max],
                cmin=0,
                cmax=colour_max,
                colorscale=scale,
                showscale=True,
                colorbar=dict(
                    title=dict(text=units, side="right"),
                    x=bar_x,
                    y=0.50,
                    len=0.65,
                    thickness=12,
                    tickfont=dict(size=11),
                ),
            ),
            showlegend=False,
            hoverinfo="skip",
        ),
        row=1, col=col,
    )

    # Label all analysed municipalities, including unavailable values.
    labelled = map_gdf.loc[map_gdf.unit_type.eq(SUPPORTS[0])]

    for row in labelled.itertuples():
        point = row.geometry.representative_point()
        ax, ay = map_offsets.get(row.unit_name, (40, -25))

        fig.add_annotation(
            x=point.x,
            y=point.y,
            text=row.unit_name,
            showarrow=True,
            arrowhead=2,
            arrowsize=0.8,
            arrowwidth=0.9,
            arrowcolor="#334155",
            ax=ax,
            ay=ay,
            font=dict(size=15, color="#243B5A"),
            bgcolor="rgba(255,255,255,0.92)",
            borderpad=2,
            row=1, col=col,
        )

    fig.update_xaxes(
        range=[west - 0.15, east + 0.15],
        title_text="Longitude",
        ticksuffix="°E",
        tickformat=".1f",
        constrain="domain",
        row=1, col=col,
    )
    fig.update_yaxes(
        range=[south - 0.15, north + 0.15],
        title_text="Latitude",
        ticksuffix="°N",
        tickformat=".1f",
        scaleanchor=f"x{col}",
        scaleratio=map_aspect,
        constrain="domain",
        row=1, col=col,
    )

# ------------------------------------------------------------
# Static-friendly layout
# ------------------------------------------------------------

style_figure(
    fig,
    "",
)

fig.update_layout(
    width=1800,
    height=800,
    margin=dict(l=90, r=145, t=115, b=165),
    legend=dict(
        orientation="h",
        x=0,
        y=-0.17,
        xanchor="left",
        yanchor="top",
        font=dict(size=13),
    ),
)

fig.update_xaxes(
    title_text="Median pixel baseline radiance<br>(nW cm⁻² sr⁻¹)",
    rangemode="tozero",
    row=1, col=1,
)
fig.update_yaxes(
    title_text="Pre-event trajectory MAD<br>(percentage points)",
    rangemode="tozero",
    row=1, col=1,
)

finish_figure(fig, "05_baseline_variability")

# Optional high-resolution static export; requires Kaleido.
# fig.write_image(
#     str(FIGURE_DIR / "05_baseline_variability.png"),
#     width=1800, height=950, scale=2,
# )

## 3. Compile the recovery trajectories

Each row is one study area at one spatial support. Each column is a non-overlapping 20-day interval during days 0–179 after Haiyan. A feature is the median of at least two admissible four-day composites. Counts accompany every value; blanks mean insufficient observation, not zero recovery.

Both supports retain the existing matched-baseline percentages. These features describe NTL relative to baseline, not the fraction of disaster damage repaired.


In [ ]:
trajectories = pd.concat(
    [municipality_four_day, kernel_four_day],
    ignore_index=True,
)

support_order = ["Municipality-wide G3", "Local 5×5 window"]
profile_index = pd.MultiIndex.from_product(
    [support_order, STUDY_AREAS],
    names=["unit_type", "unit_name"],
)

# Keep the existing 180-day clustering window.
cluster_blocks = np.arange(
    CLUSTER_HORIZON_DAYS // AGGREGATION_DAYS
)
feature_days = (
    cluster_blocks * AGGREGATION_DAYS
    + (AGGREGATION_DAYS - 1) / 2
)

selected = trajectories.loc[
    trajectories.block.isin(cluster_blocks)
].copy()

assert not selected.duplicated(
    ["unit_type", "unit_name", "block"]
).any(), "Duplicate four-day composites found."

clustering_features = (
    selected.pivot(
        index=["unit_type", "unit_name"],
        columns="block",
        values="recovery_pct",
    )
    .reindex(index=profile_index, columns=cluster_blocks)
    .replace([np.inf, -np.inf], np.nan)
)

# Here 1 means an admissible four-day composite exists.
# It is not the number of daily observations within that composite.
clustering_counts = clustering_features.notna().astype(int)

municipality_features = clustering_features.xs(support_order[0])
municipality_counts = clustering_counts.xs(support_order[0])

for support, slug in zip(support_order, ["municipality", "5x5"]):
    values = clustering_features.xs(support)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=[
            "Observed four-day NTL",
            "Four-day observation availability",
        ],
        horizontal_spacing=0.16,
    )

    fig.add_trace(
        go.Heatmap(
            x=feature_days,
            y=values.index,
            z=values.values,
            colorscale="Viridis",
            colorbar=dict(title="% baseline", x=0.43, thickness=12),
            hoverongaps=False,
            hovertemplate=(
                "%{y}<br>Day %{x:.1f}"
                "<br>%{z:.1f}% baseline<extra></extra>"
            ),
        ),
        row=1, col=1,
    )

    fig.add_trace(
        go.Heatmap(
            x=feature_days,
            y=values.index,
            z=values.notna().astype(int).values,
            zmin=0, zmax=1,
            colorscale=[[0, "#EEEEEE"], [1, "#174A7E"]],
            colorbar=dict(
                tickvals=[0, 1],
                ticktext=["Missing", "Observed"],
                thickness=12,
            ),
            hovertemplate=(
                "%{y}<br>Day %{x:.1f}"
                "<br>Available: %{z}<extra></extra>"
            ),
        ),
        row=1, col=2,
    )

    fig.update_xaxes(title_text="Days since Haiyan")
    fig.update_yaxes(autorange="reversed")
    style_figure(fig, f"Four-day clustering inputs · {support}")
    finish_figure(fig, f"clustering_matrix_{slug}")

## 4–5. Preprocess transparently and inspect candidate k

Clustering uses complete rows only: all nine intervals must meet the observation-count rule. No interpolation, extrapolation, zero filling, or within-trajectory standardisation is applied. Equal-duration features share the same percentage scale, so amplitude and persistent deficits remain part of the distance. The eligibility table records every retained and excluded support; complete cases may represent the more observable locations.

Fit the two supports separately. Inspect inertia (the elbow) and mean silhouette together. The elbow is the largest downward departure from the chord joining the first and last normalised inertia values; fewer than three candidate k values cannot identify an elbow. Select the highest silhouette among partitions without singletons when available; otherwise retain the highest finite score as an explicitly exploratory solution. A disagreement between diagnostics remains visible. Neither diagnostic establishes a unique or externally validated optimum.


In [ ]:
MIN_OBSERVED_SHARE = 0.60
MAX_INTERNAL_GAP_BLOCKS = 2
MIN_SHARED_PERIODS = 15
MIN_PERIODS_PER_STAGE = 3
MAX_K = 6

eligibility = pd.DataFrame(index=profile_index)
eligibility["observed_periods"] = clustering_counts.sum(axis=1)
eligibility["expected_periods"] = len(cluster_blocks)
eligibility["observed_share"] = (
    eligibility.observed_periods / len(cluster_blocks)
)
eligibility["eligible"] = (
    eligibility.observed_share >= MIN_OBSERVED_SHARE
)
eligibility["interpolated_periods"] = 0
eligibility["shared_periods"] = 0
eligibility["exclusion_reason"] = np.where(
    eligibility.eligible,
    "",
    f"Observed coverage below {MIN_OBSERVED_SHARE:.0%}",
)

prepared_features = clustering_features.copy()
interpolated_mask = clustering_features.notna() & False

for key, curve in clustering_features.iterrows():
    missing = curve.isna()
    run_id = missing.ne(missing.shift()).cumsum()
    run_length = missing.groupby(run_id).transform("sum")

    # Interpolation is used only in the clustering matrix.
    candidate = curve.interpolate(method="linear", limit_area="inside")
    fillable = (
        missing
        & run_length.le(MAX_INTERNAL_GAP_BLOCKS)
        & candidate.notna()
    )

    prepared_features.loc[key] = curve.where(~fillable, candidate)
    interpolated_mask.loc[key] = fillable
    eligibility.loc[key, "interpolated_periods"] = int(fillable.sum())

assignments = eligibility.reset_index()
assignments["family_id"] = np.nan
assignments["family"] = "Not grouped"
assignments["clustering_status"] = "Insufficient observed coverage"

clustering_models = {}
diagnostic_rows = []

for support, prefix in zip(support_order, ["M", "K"]):
    support_eligibility = eligibility.xs(support)
    names = support_eligibility.index[
        support_eligibility.eligible
    ]

    prepared = prepared_features.xs(support).loc[names]

    # Keep all coverage-qualified locations; select shared time columns.
    shared_blocks = (
        prepared.columns[prepared.notna().all(axis=0)]
        if len(prepared)
        else pd.Index([], dtype=int)
    )
    features = prepared.loc[:, shared_blocks]

    stage_counts = [
        int((
            (shared_blocks * AGGREGATION_DAYS >= start)
            & (shared_blocks * AGGREGATION_DAYS < start + 60)
        ).sum())
        for start in (0, 60, 120)
    ]

    support_mask = assignments.unit_type.eq(support)
    eligible_mask = support_mask & assignments.eligible

    assignments.loc[support_mask, "shared_periods"] = len(shared_blocks)
    eligibility.loc[(support, slice(None)), "shared_periods"] = len(shared_blocks)

    result = {
        "features": features,
        "shared_blocks": shared_blocks,
        "chosen_k": None,
        "elbow_k": np.nan,
        "model": None,
    }
    clustering_models[support] = result

    reason = None
    if len(features) < 3:
        reason = "Fewer than three coverage-qualified locations"
    elif (
        len(shared_blocks) < MIN_SHARED_PERIODS
        or min(stage_counts) < MIN_PERIODS_PER_STAGE
    ):
        reason = "Insufficient shared periods across the recovery window"
    elif len(np.unique(features.to_numpy(float), axis=0)) < 2:
        reason = "Fewer than two distinct trajectories"

    if reason is not None:
        result["status"] = reason
        assignments.loc[
            eligible_mask, ["clustering_status", "exclusion_reason"]
        ] = reason
        continue

    # Same units throughout: no within-trajectory standardisation.
    X = features.to_numpy(float)
    n_unique = len(np.unique(X, axis=0))
    k_max = min(MAX_K, len(X) - 1, n_unique)

    models = {}
    rows = []

    for k in range(1, k_max + 1):
        fitted = KMeans(
            n_clusters=k,
            n_init=20,
            random_state=CLUSTER_RANDOM_STATE,
        ).fit(X)

        labels = fitted.labels_
        _, sizes = np.unique(labels, return_counts=True)

        models[k] = fitted
        rows.append({
            "unit_type": support,
            "k": k,
            "inertia": fitted.inertia_,
            "silhouette": (
                silhouette_score(X, labels)
                if 1 < len(sizes) < len(X)
                else np.nan
            ),
            "minimum_family_size": int(sizes.min()),
        })

    diagnostic = pd.DataFrame(rows)

    # Simple elbow diagnostic: departure from normalised endpoint chord.
    elbow_k = np.nan
    inertia_range = diagnostic.inertia.iloc[0] - diagnostic.inertia.iloc[-1]

    if len(diagnostic) >= 3 and inertia_range > 0:
        x = (
            (diagnostic.k - diagnostic.k.iloc[0])
            / (diagnostic.k.iloc[-1] - diagnostic.k.iloc[0])
        )
        y = (
            (diagnostic.inertia - diagnostic.inertia.iloc[-1])
            / inertia_range
        )
        departure = ((1 - x) - y).iloc[1:-1]

        if departure.max() > 1e-9:
            elbow_k = int(diagnostic.loc[departure.idxmax(), "k"])

    scored = diagnostic.dropna(subset=["silhouette"])
    non_singleton = scored.loc[scored.minimum_family_size.ge(2)]
    candidates = non_singleton if not non_singleton.empty else scored

    if candidates.empty:
        result["status"] = "No valid silhouette comparison"
        assignments.loc[
            eligible_mask, ["clustering_status", "exclusion_reason"]
        ] = result["status"]
        diagnostic["chosen_k"] = np.nan
    else:
        chosen = candidates.sort_values(
            ["silhouette", "k"], ascending=[False, True]
        ).iloc[0]
        selected_k = int(chosen.k)
        fitted = models[selected_k]

        centre_order = np.argsort(
            fitted.cluster_centers_.mean(axis=1), kind="stable"
        )
        label_map = {
            int(old): new for new, old in enumerate(centre_order)
        }
        labels = np.array([label_map[int(v)] for v in fitted.labels_])

        status = "Exploratory solution"
        if len(scored) == 1:
            status += "; only one silhouette-valid k"
        if non_singleton.empty:
            status += "; singleton family present"
        if chosen.silhouette <= 0:
            status += "; no positive silhouette separation"
        if pd.isna(elbow_k):
            status += "; elbow unavailable"
        elif elbow_k != selected_k:
            status += f"; elbow instead suggests k={int(elbow_k)}"

        result.update(
            chosen_k=selected_k,
            elbow_k=elbow_k,
            model=fitted,
            labels=labels,
            status=status,
        )

        for name, label in zip(features.index, labels):
            mask = support_mask & assignments.unit_name.eq(name)
            assignments.loc[
                mask, ["family_id", "family", "clustering_status"]
            ] = [int(label), f"{prefix}{label + 1}", status]

        diagnostic["chosen_k"] = selected_k

    diagnostic["elbow_k"] = elbow_k
    diagnostic_rows.extend(diagnostic.to_dict("records"))

diagnostics_all = pd.DataFrame(
    diagnostic_rows,
    columns=[
        "unit_type", "k", "inertia", "silhouette",
        "minimum_family_size", "chosen_k", "elbow_k",
    ],
)

# Preserve names used by the existing export section.
complete_features = clustering_models[support_order[0]]["features"]
complete_features_all = pd.concat(
    {support: result["features"]
     for support, result in clustering_models.items()},
    names=["unit_type", "unit_name"],
)
family_id = (
    assignments.loc[
        assignments.unit_type.eq(support_order[0])
        & assignments.family_id.notna()
    ]
    .set_index("unit_name").family_id.astype(int)
)
chosen_k = clustering_models[support_order[0]]["chosen_k"]

display(assignments[[
    "unit_type", "unit_name", "observed_periods",
    "interpolated_periods", "shared_periods",
    "family", "exclusion_reason",
]])

In [ ]:
# Evidence categories:
# 0 = missing / not used
# 1 = observed, but excluded from the comparison
# 2 = observed and used
# 3 = interpolated and used

evidence_colours = [
    "#F1F3F5",
    "#A9B6C5",
    "#174A7E",
    "#E69F00",
]
evidence_labels = [
    "Missing / not used",
    "Observed / not used",
    "Observed / used",
    "Interpolated / used",
]
evidence_scale = []
for i, colour in enumerate(evidence_colours):
    evidence_scale.extend([
        [i / 4, colour],
        [(i + 1) / 4, colour],
    ])

for support, slug in zip(support_order, ["municipality", "5x5"]):
    result = clustering_models[support]
    diagnostic = diagnostics_all.loc[
        diagnostics_all.unit_type.eq(support)
    ].sort_values("k")

    print(f"{support}: {result['status']}")

    # --------------------------------------------------------
    # Observation evidence behind the clustering matrix
    # --------------------------------------------------------

    original = (
        clustering_features.xs(support)
        .reindex(index=STUDY_AREAS, columns=cluster_blocks)
    )

    # Preserve observed values even when they were not selected.
    evidence = original.notna().astype(int)
    names = result["features"].index
    shared = result["shared_blocks"]

    if len(names) and len(shared):
        filled = (
            interpolated_mask.xs(support)
            .loc[names, shared]
            .astype(int)
        )
        evidence.loc[names, shared] = 2 + filled

    observed_used = evidence.eq(2).sum(axis=1)
    interpolated_used = evidence.eq(3).sum(axis=1)

    row_labels = [
        (
            f"{name}   "
            f"[{observed_used.loc[name]} observed"
            f" + {interpolated_used.loc[name]} filled]"
        )
        for name in STUDY_AREAS
    ]

    hover_status = np.array(
        evidence_labels, dtype=object
    )[evidence.to_numpy()]

    fig = go.Figure(go.Heatmap(
        x=feature_days,
        y=STUDY_AREAS,
        z=evidence.values,
        zmin=-0.5,
        zmax=3.5,
        colorscale=evidence_scale,
        showscale=False,
        xgap=1,
        ygap=3,
        customdata=hover_status,
        hovertemplate=(
            "<b>%{y}</b>"
            "<br>Four-day midpoint: day %{x:.1f}"
            "<br>%{customdata}"
            "<extra></extra>"
        ),
    ))

    # Discrete legend, instead of a continuous-looking colourbar.
    for label, colour in zip(evidence_labels, evidence_colours):
        fig.add_trace(go.Scatter(
            x=[None],
            y=[None],
            mode="markers",
            marker=dict(
                symbol="square",
                size=13,
                color=colour,
                line=dict(color="#CBD5E1", width=0.6),
            ),
            name=label,
            hoverinfo="skip",
        ))

    fig.add_vline(
        x=0,
        line_dash="dash",
        line_color=EVENT_COLOR,
        line_width=1.5,
    )

    style_figure(
        fig,
        f"Four-day comparison periods | {support}",
    )
    fig.update_layout(
        height=400,
        margin=dict(l=285, r=35, t=95, b=105),
        legend=dict(
            orientation="h",
            x=0.3,
            y=1.15,
            xanchor="left",
            yanchor="top",
        ),
    )
    fig.update_yaxes(
        autorange="reversed",
        tickmode="array",
        tickvals=STUDY_AREAS,
        ticktext=row_labels,
        showgrid=False,
        tickfont=dict(size=12),
    )
    fig.update_xaxes(
        title_text="Days since Haiyan (4D tiles)",
        range=[-2, CLUSTER_HORIZON_DAYS],
        tick0=0,
        dtick=20,  # Tick spacing only; the data remain four-day features.
        showgrid=False,
    )

    finish_figure(fig, f"clustering_used_periods_{slug}")

    # --------------------------------------------------------
    # Elbow and silhouette diagnostics
    # --------------------------------------------------------

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=[
            "Within-family variation",
            "Separation between families",
        ],
        horizontal_spacing=0.05,
    )

    if diagnostic.empty:
        fig.add_annotation(
            x=0.5,
            y=0.5,
            xref="paper",
            yref="paper",
            text="No model fitted<br>" + result["status"],
            showarrow=False,
            font=dict(size=15, color="#64748B"),
        )

    else:
        for col, metric in [(1, "inertia"), (2, "silhouette")]:
            plotted = diagnostic.dropna(subset=[metric])

            marker_colours = [
                "#D55E00" if k == result["chosen_k"] else "#174A7E"
                for k in plotted.k
            ]
            value_labels = [
                f"{value:,.0f}" if metric == "inertia" else f"{value:.3f}"
                for value in plotted[metric]
            ]

            fig.add_trace(
                go.Scatter(
                    x=plotted.k,
                    y=plotted[metric],
                    mode="lines+markers+text",
                    text=value_labels,
                    textposition="top center",
                    textfont=dict(size=12),
                    cliponaxis=False,
                    line=dict(color="#8193A5", width=2),
                    marker=dict(
                        color=marker_colours,
                        size=12,
                        line=dict(color="white", width=1),
                    ),
                    showlegend=False,
                    customdata=plotted[["minimum_family_size"]],
                    hovertemplate=(
                        "<b>k = %{x}</b>"
                        "<br>Value: %{y:.3f}"
                        "<br>Smallest family: %{customdata[0]}"
                        "<extra></extra>"
                    ),
                ),
                row=1, col=col,
            )

            if result["chosen_k"] is not None:
                fig.add_vline(
                    x=result["chosen_k"],
                    line_dash="dash",
                    line_color="#D55E00",
                    line_width=1.5,
                    row=1, col=col,
                )

        if pd.notna(result["elbow_k"]):
            fig.add_vline(
                x=result["elbow_k"],
                line_dash="dot",
                line_color="#009E73",
                line_width=2,
                row=1, col=1,
            )

        fig.update_xaxes(
            range=[
                diagnostic.k.min() - 0.35,
                diagnostic.k.max() + 0.35,
            ],
        )

        # Explicitly flag the earlier three-location problem.
        if diagnostic.silhouette.notna().sum() == 1:
            fig.add_annotation(
                x=0.5,
                y=0.10,
                xref="x2 domain",
                yref="y2 domain",
                text="Only one valid k: no comparison of alternatives",
                showarrow=False,
                font=dict(size=12, color="#D55E00"),
                bgcolor="rgba(255,255,255,0.9)",
            )

    fig.add_hline(
        y=0,
        line_color="#AAB4BE",
        line_dash="dot",
        row=1, col=2,
    )

    style_figure(
        fig,
        f"Candidate family counts | {support}",
    )
    fig.update_layout(
        height=400,
        margin=dict(l=90, r=50, t=110, b=100),
    )
    fig.update_xaxes(
        title_text="Number of families (k)",
        dtick=1,
    )
    fig.update_yaxes(
        title_text="Inertia",
        rangemode="tozero",
        row=1, col=1,
    )
    fig.update_yaxes(
        title_text="Mean silhouette",
        range=[0.15, 0.35],
        row=1, col=2,
    )

    finish_figure(fig, f"clustering_k_diagnostics_{slug}")

## 6. Describe the candidate trajectory families

Thin curves retain individual trajectories; thick dashed curves show k-means means. The companion panels show pointwise medians and interquartile envelopes across the same members. These are functional-style descriptive summaries, **not formal depth-based functional boxplots or uncertainty intervals**. A singleton has no between-member variability to summarise.

M1, M2, … identify municipality families; K1, K2, … identify local 5×5 families. Within each support, numbers run from lower to higher mean baseline-relative NTL. The independently fitted families are support-specific; matching numbers or colours do not establish equivalent membership or recovery behaviour.


In [ ]:
summary_rows = []

for support, slug in zip(support_order, ["municipality", "5x5"]):
    result = clustering_models[support]

    fig = make_subplots(
        rows=1,
        cols=3,
        column_widths=[0.35, 0.35, 0.30],
        horizontal_spacing=0.065,
        subplot_titles=[
            "Observed members and family median",
            "Family median and pointwise IQR",
            "Spatial distribution",
        ],
    )

    # --------------------------------------------------------
    # Temporal panels
    # --------------------------------------------------------

    if result["chosen_k"] is None:
        fig.add_annotation(
            x=0.32, y=0.5,
            xref="paper", yref="paper",
            text=result["status"],
            showarrow=False,
        )

    else:
        for family in range(result["chosen_k"]):
            members = assignments.loc[
                assignments.unit_type.eq(support)
                & assignments.family_id.eq(family)
            ]
            names = members.unit_name.tolist()
            label = members.family.iloc[0]
            colour = FAMILY_COLORS[family]
            values = result["features"].loc[names]

            mean = values.mean().reindex(cluster_blocks)
            quantiles = values.quantile([0.25, 0.50, 0.75])
            q25 = quantiles.loc[0.25].reindex(cluster_blocks)
            median = quantiles.loc[0.50].reindex(cluster_blocks)
            q75 = quantiles.loc[0.75].reindex(cluster_blocks)

            rgb = tuple(
                int(colour.lstrip("#")[i:i + 2], 16)
                for i in (0, 2, 4)
            )
            fill_colour = (
                f"rgba({rgb[0]},{rgb[1]},{rgb[2]},0.16)"
            )

            # Individual observed profiles.
            for name in names:
                observed = (
                    clustering_features.loc[(support, name)]
                    .reindex(cluster_blocks)
                )

                fig.add_trace(
                    go.Scatter(
                        x=feature_days,
                        y=observed,
                        mode="lines",
                        connectgaps=False,
                        line=dict(color=colour, width=1),
                        opacity=0.30,
                        name=f"{label} · {name}",
                        legendgroup=label,
                        showlegend=False,
                        hovertemplate=(
                            f"{name}<br>Day %{{x:.1f}}"
                            "<br>%{y:.1f}% baseline<extra></extra>"
                        ),
                    ),
                    row=1, col=1,
                )

            # Draw each continuous IQR segment independently.
            # Explicit polygons cannot bridge excluded periods.
            valid_positions = np.flatnonzero(
                q25.notna().to_numpy()
                & q75.notna().to_numpy()
            )

            runs = (
                np.split(
                    valid_positions,
                    np.where(
                        np.diff(
                            np.asarray(cluster_blocks)[valid_positions]
                        ) != 1
                    )[0] + 1,
                )
                if len(valid_positions)
                else []
            )

            if len(names) > 1:
                for run in runs:
                    x = np.asarray(feature_days)[run]
                    low = q25.iloc[run].to_numpy()
                    high = q75.iloc[run].to_numpy()

                    if len(run) == 1:
                        # A single retained period has no band width.
                        fig.add_trace(
                            go.Scatter(
                                x=[x[0], x[0]],
                                y=[low[0], high[0]],
                                mode="lines",
                                line=dict(color=colour, width=2),
                                opacity=0.35,
                                legendgroup=label,
                                showlegend=False,
                                hoverinfo="skip",
                            ),
                            row=1, col=2,
                        )
                    else:
                        fig.add_trace(
                            go.Scatter(
                                x=np.concatenate([x, x[::-1]]),
                                y=np.concatenate([low, high[::-1]]),
                                mode="lines",
                                line=dict(width=0),
                                fill="toself",
                                fillcolor=fill_colour,
                                legendgroup=label,
                                showlegend=False,
                                hoverinfo="skip",
                            ),
                            row=1, col=2,
                        )

            # Same prepared-family median in both temporal panels.
            for col in (1, 2):
                fig.add_trace(
                    go.Scatter(
                        x=feature_days,
                        y=median,
                        mode="lines+markers",
                        connectgaps=False,
                        line=dict(color=colour, width=2.8),
                        marker=dict(
                            size=5,
                            color=colour,
                            line=dict(color="white", width=0.5),
                        ),
                        name=f"{label} (n={len(names)})",
                        legendgroup=label,
                        showlegend=(col == 1),
                        hovertemplate=(
                            f"{label} median"
                            "<br>Day %{x:.1f}"
                            "<br>%{y:.1f}% baseline<extra></extra>"
                        ),
                    ),
                    row=1, col=col,
                )

            for block in values.columns:
                summary_rows.append({
                    "unit_type": support,
                    "family_id": family,
                    "family": label,
                    "n_members": len(names),
                    "members": ", ".join(names),
                    "block": int(block),
                    "day": (
                        int(block) * AGGREGATION_DAYS
                        + (AGGREGATION_DAYS - 1) / 2
                    ),
                    "mean_pct": mean.loc[block],
                    "median_pct": median.loc[block],
                    "q25_pct": q25.loc[block],
                    "q75_pct": q75.loc[block],
                })

    # --------------------------------------------------------
    # Spatial panel: same family colours as the trajectories
    # --------------------------------------------------------

    west, south, east, north = municipalities_display.total_bounds
    bounds = [west - 0.12, south - 0.12, east + 0.12, north + 0.12]

    regional_view = regional_boundaries.cx[
        bounds[0]:bounds[2],
        bounds[1]:bounds[3],
    ]

    context_x, context_y = [], []
    for geometry in regional_view.geometry:
        xx, yy = boundary_coordinates(geometry)
        context_x.extend(xx)
        context_y.extend(yy)

    fig.add_trace(
        go.Scatter(
            x=context_x,
            y=context_y,
            mode="lines",
            line=dict(color="#CFD6DE", width=0.6),
            showlegend=False,
            hoverinfo="skip",
        ),
        row=1, col=3,
    )

    lookup = (
        assignments.loc[assignments.unit_type.eq(support)]
        .set_index("unit_name")
    )
    ungrouped_legend_added = False

    for row in municipalities_display.itertuples():
        assignment = lookup.loc[row.unit_name]
        grouped = pd.notna(assignment.family_id)

        colour = (
            FAMILY_COLORS[int(assignment.family_id)]
            if grouped else "#9BA3AC"
        )
        label = assignment.family if grouped else "Not grouped"

        # Municipal families fill the boundary.
        # Local families fill only their actual 5×5 window.
        geometry = (
            row.geometry
            if support == support_order[0]
            else kernel_windows.get(row.unit_name)
        )

        point = (
            geometry.representative_point()
            if geometry is not None
            else row.geometry.representative_point()
        )

        if geometry is not None:
            polygons = (
                list(geometry.geoms)
                if geometry.geom_type == "MultiPolygon"
                else [geometry]
            )

            for polygon in polygons:
                xx, yy = boundary_coordinates(polygon)

                fig.add_trace(
                    go.Scatter(
                        x=xx,
                        y=yy,
                        mode="lines",
                        fill="toself",
                        fillcolor=colour,
                        opacity=0.35,
                        line=dict(color=colour, width=1.5),
                        legendgroup=label,
                        showlegend=False,
                        hoverinfo="skip",
                    ),
                    row=1, col=3,
                )

        fig.add_trace(
            go.Scatter(
                x=[point.x],
                y=[point.y],
                mode="markers+text",
                text=[row.unit_name],
                textposition="top center",
                textfont=dict(size=10, color="#243B5A"),
                marker=dict(
                    color=colour,
                    size=8,
                    line=dict(color="white", width=1),
                ),
                name=label,
                legendgroup=label,
                showlegend=(
                    not grouped and not ungrouped_legend_added
                ),
                hovertemplate=(
                    f"<b>{row.unit_name}</b><br>{label}"
                    f"<br>{assignment.clustering_status}"
                    "<extra></extra>"
                ),
            ),
            row=1, col=3,
        )

        if not grouped:
            ungrouped_legend_added = True

    # --------------------------------------------------------
    # Layout
    # --------------------------------------------------------

    for col in (1, 2):
        fig.add_hline(
            y=100,
            line_dash="dot",
            line_color=BASELINE_COLOR,
            row=1, col=col,
        )
        fig.add_vline(
            x=0,
            line_dash="dash",
            line_color=EVENT_COLOR,
            row=1, col=col,
        )
        fig.update_xaxes(
            title_text="Days since Haiyan",
            range=[-3, CLUSTER_HORIZON_DAYS],
            dtick=30,
            row=1, col=col,
        )
        fig.update_yaxes(
            title_text="NTL (% matched baseline)",
            rangemode="tozero",
            row=1, col=col,
        )

    # Match only the two temporal axes, not the map.
    fig.update_yaxes(matches="y", row=1, col=2)

    fig.update_xaxes(
        title_text="Longitude",
        range=[bounds[0], bounds[2]],
        ticksuffix="°E",
        tickformat=".1f",
        constrain="domain",
        row=1, col=3,
    )
    fig.update_yaxes(
        title_text="Latitude",
        range=[bounds[1], bounds[3]],
        ticksuffix="°N",
        tickformat=".1f",
        scaleanchor="x3",
        scaleratio=1 / np.cos(np.deg2rad((south + north) / 2)),
        constrain="domain",
        row=1, col=3,
    )

    style_figure(
        fig,
        f"Candidate recovery families | {support}",
    )
    fig.update_layout(
        width=1800,
        height=600,
        margin=dict(l=85, r=45, t=110, b=150),
        legend=dict(
            orientation="h",
            x=0,
            y=-0.15,
            xanchor="left",
            yanchor="top",
        ),
    )

    finish_figure(fig, f"clustering_family_trajectories_{slug}")

family_summaries = pd.DataFrame(
    summary_rows,
    columns=[
        "unit_type", "family_id", "family", "n_members",
        "members", "block", "day", "mean_pct",
        "median_pct", "q25_pct", "q75_pct",
    ],
)


In [ ]:
# ============================================================
# PIXEL-LEVEL RECOVERY FAMILIES
# Four-day G3 trajectories · raw P95-capped NTL
# ============================================================

from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd
import xarray as xr
import plotly.graph_objects as go


# ------------------------------------------------------------
# 1. Build four-day trajectories for every G3 pixel
# ------------------------------------------------------------

PIXEL_HORIZON_DAYS = 180
PIXEL_AGGREGATION_DAYS = 4
PIXEL_MIN_VALID_COMPOSITES = 2
PIXEL_MAX_K = 8
PIXEL_RANDOM_STATE = 42
PIXEL_FIT_SAMPLE = 10000
PIXEL_SILHOUETTE_SAMPLE = 3000

pixel_blocks = np.arange(
    0,
    PIXEL_HORIZON_DAYS // PIXEL_AGGREGATION_DAYS,
)

pixel_dates = pd.DatetimeIndex(rq_cube.date.values).normalize()
pixel_block_ids = (
    (pixel_dates - EVENT_DATE).days
    // PIXEL_AGGREGATION_DAYS
)

# rq_cube already contains:
# - DNB_BRDF_Corrected_NTL
# - MQF == 0 observations
# - the existing daily 95th-percentile cap
# - G3 support from the current processing pipeline
pixel_signal = rq_cube

pixel_composites = (
    pixel_signal
    .sel(
        date=slice(
            EVENT_DATE,
            EVENT_DATE + pd.Timedelta(days=PIXEL_HORIZON_DAYS - 1),
        )
    )
    .assign_coords(
        block=("date", pixel_block_ids[
            (pixel_dates >= EVENT_DATE)
            & (
                pixel_dates
                <= EVENT_DATE + pd.Timedelta(
                    days=PIXEL_HORIZON_DAYS - 1
                )
            )
        ])
    )
    .groupby("block")
    .median("date", skipna=True)
    .reindex(block=pixel_blocks)
)

# Stack y/x into individual pixels.
pixel_matrix = (
    pixel_composites
    .stack(pixel=("y", "x"))
    .transpose("pixel", "block")
)

with xr.set_options(keep_attrs=True):
    pixel_matrix = pixel_matrix.compute()

pixel_values = np.asarray(
    pixel_matrix.to_numpy(),
    dtype=float,
)
pixel_index = pixel_matrix.pixel.to_index()

pixel_xy = pd.DataFrame({
    "pixel": np.arange(len(pixel_index)),
    "y": pixel_index.get_level_values("y").to_numpy(),
    "x": pixel_index.get_level_values("x").to_numpy(),
})

pixel_xy["longitude"] = pixel_xy.x.map(
    dict(enumerate(pixel_composites.x.values))
)
pixel_xy["latitude"] = pixel_xy.y.map(
    dict(enumerate(pixel_composites.y.values))
)

pixel_observed = np.isfinite(pixel_values)
pixel_valid_count = pixel_observed.sum(axis=1)

pixel_eligible = (
    pixel_valid_count >= PIXEL_MIN_VALID_COMPOSITES
)

pixel_values = pixel_values[pixel_eligible]
pixel_xy = pixel_xy.loc[pixel_eligible].reset_index(drop=True)

print(
    f"Eligible G3 pixels: {len(pixel_values):,}"
    f" / {len(pixel_matrix.pixel):,}"
)


# ------------------------------------------------------------
# 2. Interpolate gaps for clustering only
# ------------------------------------------------------------

pixel_features = pd.DataFrame(
    pixel_values,
    columns=pixel_blocks,
)

# Linear interpolation across internal gaps.
# Edge gaps are filled with the nearest observed trajectory value
# so that every eligible pixel can be compared.
pixel_features = (
    pixel_features
    .interpolate(
        axis=1,
        method="linear",
        limit_direction="both",
    )
)

# Any trajectory that remains invalid is removed.
feature_mask = np.isfinite(pixel_features.to_numpy()).all(axis=1)

pixel_features = pixel_features.loc[feature_mask].reset_index(drop=True)
pixel_xy = pixel_xy.loc[feature_mask].reset_index(drop=True)

X = pixel_features.to_numpy(dtype="float32")

print(
    f"Pixel trajectories used for clustering: {len(X):,}"
    f" × {X.shape[1]} four-day periods"
)


# ------------------------------------------------------------
# 3. Elbow and silhouette diagnostics
# ------------------------------------------------------------

rng = np.random.default_rng(PIXEL_RANDOM_STATE)

fit_size = min(PIXEL_FIT_SAMPLE, len(X))
fit_indices = rng.choice(
    len(X),
    size=fit_size,
    replace=False,
)

X_fit = X[fit_indices]

diagnostic_rows = []
pixel_models = {}

for k in range(
    1,
    min(PIXEL_MAX_K, len(X_fit) - 1) + 1,
):
    model = MiniBatchKMeans(
        n_clusters=k,
        random_state=PIXEL_RANDOM_STATE,
        n_init=10,
        batch_size=2048,
        max_iter=300,
    )

    labels = model.fit_predict(X_fit)
    pixel_models[k] = model

    silhouette = np.nan

    if k >= 2 and len(np.unique(labels)) < len(labels):
        silhouette_indices = rng.choice(
            len(X_fit),
            size=min(PIXEL_SILHOUETTE_SAMPLE, len(X_fit)),
            replace=False,
        )

        silhouette = silhouette_score(
            X_fit[silhouette_indices],
            labels[silhouette_indices],
        )

    family_sizes = np.bincount(labels)

    diagnostic_rows.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette": silhouette,
        "minimum_family_size": family_sizes.min(),
    })

pixel_diagnostics = pd.DataFrame(diagnostic_rows)

# Normalised elbow departure from the first-to-last inertia chord.
pixel_elbow_k = np.nan

if len(pixel_diagnostics) >= 3:
    inertia = pixel_diagnostics.inertia.to_numpy()
    k_values = pixel_diagnostics.k.to_numpy()

    x_norm = (
        k_values - k_values.min()
    ) / (
        k_values.max() - k_values.min()
    )

    y_norm = (
        inertia - inertia[-1]
    ) / (
        inertia[0] - inertia[-1]
    )

    elbow_distance = ((1 - x_norm) - y_norm)

    interior = np.arange(1, len(elbow_distance) - 1)

    if len(interior):
        elbow_index = interior[
            np.argmax(elbow_distance[interior])
        ]
        pixel_elbow_k = int(k_values[elbow_index])

valid_silhouette = pixel_diagnostics.dropna(
    subset=["silhouette"]
)

if valid_silhouette.empty:
    raise ValueError(
        "No valid silhouette scores were produced."
    )

pixel_selected_k = int(
    valid_silhouette
    .sort_values(
        ["silhouette", "k"],
        ascending=[False, True],
    )
    .iloc[0]
    .k
)

pixel_model = pixel_models[pixel_selected_k]

print(
    f"Selected k: {pixel_selected_k}"
    f" | elbow estimate: {pixel_elbow_k}"
)


# ------------------------------------------------------------
# 4. Plot elbow and silhouette diagnostics
# ------------------------------------------------------------

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Elbow diagnostic",
        "Silhouette diagnostic",
    ],
    horizontal_spacing=0.14,
)

fig.add_trace(
    go.Scatter(
        x=pixel_diagnostics.k,
        y=pixel_diagnostics.inertia,
        mode="lines+markers+text",
        text=[
            f"{value:,.0f}"
            for value in pixel_diagnostics.inertia
        ],
        textposition="top center",
        marker=dict(
            size=11,
            color=[
                "#D55E00"
                if k == pixel_selected_k
                else "#174A7E"
                for k in pixel_diagnostics.k
            ],
        ),
        line=dict(color="#8092A5", width=2),
        name="Inertia",
        showlegend=False,
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=pixel_diagnostics.k,
        y=pixel_diagnostics.silhouette,
        mode="lines+markers+text",
        text=[
            ""
            if pd.isna(value)
            else f"{value:.3f}"
            for value in pixel_diagnostics.silhouette
        ],
        textposition="top center",
        marker=dict(
            size=11,
            color=[
                "#D55E00"
                if k == pixel_selected_k
                else "#174A7E"
                for k in pixel_diagnostics.k
            ],
        ),
        line=dict(color="#8092A5", width=2),
        name="Silhouette",
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig.add_vline(
    x=pixel_selected_k,
    line_dash="dash",
    line_color="#D55E00",
    row=1,
    col=1,
)

fig.add_vline(
    x=pixel_selected_k,
    line_dash="dash",
    line_color="#D55E00",
    row=1,
    col=2,
)

if pd.notna(pixel_elbow_k):
    fig.add_vline(
        x=pixel_elbow_k,
        line_dash="dot",
        line_color="#009E73",
        row=1,
        col=1,
    )

fig.update_xaxes(
    title_text="Number of pixel families (k)",
    dtick=1,
)

fig.update_yaxes(
    title_text="Within-family inertia",
    rangemode="tozero",
    row=1,
    col=1,
)

fig.update_yaxes(
    title_text="Mean silhouette",
    range=[-1, 1],
    row=1,
    col=2,
)

style_figure(
    fig,
    "Pixel-level recovery-family diagnostics",
)

fig.update_layout(
    width=1400,
    height=400,
    margin=dict(l=90, r=50, t=105, b=100),
)

finish_figure(
    fig,
    "pixel_family_k_diagnostics",
)



In [ ]:

# ------------------------------------------------------------
# 5. Predict family labels for all eligible pixels
# ------------------------------------------------------------

pixel_labels_raw = pixel_model.predict(X)

# Order families by their mean raw trajectory level.
family_order = np.argsort(
    pixel_model.cluster_centers_.mean(axis=1)
)

family_label_map = {
    int(old): int(new)
    for new, old in enumerate(family_order)
}

pixel_family_id = np.array([
    family_label_map[int(label)]
    for label in pixel_labels_raw
])

pixel_xy["family_id"] = pixel_family_id
pixel_xy["family"] = pixel_xy.family_id.map(
    lambda value: f"F{int(value) + 1}"
)

pixel_family_sizes = (
    pixel_xy
    .groupby(["family_id", "family"])
    .size()
    .rename("pixels")
    .reset_index()
)

display(pixel_family_sizes)


In [ ]:


# ------------------------------------------------------------
# 6. Functional-style boxplots
# ------------------------------------------------------------

# For readability, show up to 250 randomly selected pixels per family.
# The fitted family median uses every assigned pixel.
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Pixel trajectories by family",
        "Functional-style family summaries",
    ],
    shared_yaxes=True,
)

for family in range(pixel_selected_k):
    family_name = f"F{family + 1}"
    colour = FAMILY_COLORS[family]

    members = np.flatnonzero(
        pixel_family_id == family
    )

    values = X[members]

    display_members = members

    if len(display_members) > 250:
        display_members = rng.choice(
            display_members,
            size=250,
            replace=False,
        )

    for member in display_members:
        fig.add_trace(
            go.Scatter(
                x=feature_days,
                y=X[member],
                mode="lines",
                connectgaps=False,
                line=dict(color=colour, width=0.7),
                opacity=0.12,
                showlegend=False,
                hoverinfo="skip",
            ),
            row=1,
            col=1,
        )

    family_median = np.nanmedian(values, axis=0)
    family_q25 = np.nanquantile(values, 0.25, axis=0)
    family_q75 = np.nanquantile(values, 0.75, axis=0)

    fig.add_trace(
        go.Scatter(
            x=feature_days,
            y=family_median,
            mode="lines+markers",
            line=dict(color=colour, width=3),
            marker=dict(size=5),
            name=f"{family_name} · n={len(values):,}",
            legendgroup=family_name,
        ),
        row=1,
        col=1,
    )

    # Pointwise IQR band.
    fig.add_trace(
        go.Scatter(
            x=np.concatenate([
                feature_days,
                feature_days[::-1],
            ]),
            y=np.concatenate([
                family_q25,
                family_q75[::-1],
            ]),
            mode="lines",
            line=dict(width=0),
            fill="toself",
            fillcolor=(
                colour.replace("rgb", "rgba")
                if colour.startswith("rgb")
                else colour
            ),
            opacity=0.14,
            showlegend=False,
            hoverinfo="skip",
            legendgroup=family_name,
        ),
        row=1,
        col=2,
    )

    fig.add_trace(
        go.Scatter(
            x=feature_days,
            y=family_median,
            mode="lines+markers",
            line=dict(color=colour, width=3),
            marker=dict(size=5),
            name=f"{family_name} median",
            legendgroup=family_name,
            showlegend=False,
        ),
        row=1,
        col=2,
    )


fig.add_vline(
    x=0,
    line_dash="dash",
    line_color="#0057FF",
    row=1,
    col=1,
)

fig.add_vline(
    x=0,
    line_dash="dash",
    line_color="#0057FF",
    row=1,
    col=2,
)

fig.update_xaxes(
    title_text="Days since Haiyan",
)

fig.update_yaxes(
    title_text="Raw P95-capped NTL",
    rangemode="tozero",
)

style_figure(
    fig,
    "Functional-style summaries of pixel recovery families",
)

fig.update_layout(
    width=1400,
    height=400,
    margin=dict(l=50, r=50, t=50, b=50),
)

finish_figure(
    fig,
    "pixel_family_functional_summaries",
)



In [ ]:
# ============================================================
# 7–8. PROJECT PIXEL FAMILIES TO SPACE
# Regional Samar–Leyte view + Tacloban, Ormoc, Guiuan zooms
# ============================================================

# pixel_xy was created from the stacked pixel coordinates.
# Do not remap these values through pixel IDs.
pixel_xy["longitude"] = pd.to_numeric(
    pixel_xy["x"],
    errors="coerce",
)
pixel_xy["latitude"] = pd.to_numeric(
    pixel_xy["y"],
    errors="coerce",
)

pixel_xy = pixel_xy.loc[
    pixel_xy.longitude.notna()
    & pixel_xy.latitude.notna()
].copy()

# Keep only family-labelled pixels.
pixel_xy["family_id"] = pixel_family_id[
    pixel_xy.index.to_numpy()
]

pixel_xy["family"] = pixel_xy.family_id.map(
    lambda value: f"F{int(value) + 1}"
)


# ------------------------------------------------------------
# Plot helpers
# ------------------------------------------------------------

def add_boundary_lines(
    figure,
    geodataframe,
    row,
    col,
    colour="#D0D7DE",
    width=0.6,
):
    for geometry in geodataframe.geometry:
        if geometry is None or geometry.is_empty:
            continue

        xx, yy = boundary_coordinates(geometry)

        figure.add_trace(
            go.Scatter(
                x=xx,
                y=yy,
                mode="lines",
                line=dict(color=colour, width=width),
                showlegend=False,
                hoverinfo="skip",
            ),
            row=row,
            col=col,
        )


def add_family_panel(
    figure,
    row,
    col,
    title,
    bounds=None,
    boundary_names=None,
    show_legend=False,
):
    panel_pixels = pixel_xy.copy()

    if bounds is not None:
        west, south, east, north = bounds

        panel_pixels = panel_pixels.loc[
            panel_pixels.longitude.between(west, east)
            & panel_pixels.latitude.between(south, north)
        ]

    # Plot the regional boundary context.
    if boundary_names is None:
        boundary_frame = regional_boundaries
    else:
        boundary_frame = municipalities_display.loc[
            municipalities_display.unit_name.isin(boundary_names)
        ]

    add_boundary_lines(
        figure,
        boundary_frame,
        row,
        col,
        colour="#CBD5E1",
        width=0.65,
    )

    # Plot every family as a separate point layer.
    for family in range(pixel_selected_k):
        subset = panel_pixels.loc[
            panel_pixels.family_id.eq(family)
        ]

        if subset.empty:
            continue

        figure.add_trace(
            go.Scattergl(
                x=subset.longitude,
                y=subset.latitude,
                mode="markers",
                marker=dict(
                    color=FAMILY_COLORS[family],
                    size=3.5,
                    opacity=0.72,
                ),
                name=f"F{family + 1}",
                legendgroup=f"F{family + 1}",
                showlegend=show_legend,
                customdata=subset[[
                    "family",
                    "pixel_id",
                ]].to_numpy()
                if "pixel_id" in subset.columns
                else None,
                hovertemplate=(
                    f"<b>F{family + 1}</b>"
                    "<br>Longitude: %{x:.5f}"
                    "<br>Latitude: %{y:.5f}"
                    "<extra></extra>"
                ),
            ),
            row=row,
            col=col,
        )

    # Add labels for zoom municipalities.
    if boundary_names is not None:
        for name in boundary_names:
            match = municipalities_display.loc[
                municipalities_display.unit_name.eq(name)
            ]

            if match.empty:
                continue

            point = match.geometry.iloc[0].representative_point()

            figure.add_annotation(
                x=point.x,
                y=point.y,
                text=f"<b>{name}</b>",
                showarrow=True,
                arrowhead=2,
                arrowsize=0.8,
                arrowwidth=0.9,
                arrowcolor="#243B5A",
                ax=25,
                ay=-25,
                bgcolor="rgba(255,255,255,0.90)",
                bordercolor="#CBD5E1",
                borderwidth=0.6,
                borderpad=3,
                font=dict(
                    family="Arial",
                    size=11,
                    color="#243B5A",
                ),
                row=row,
                col=col,
            )

    # Set panel bounds.
    if bounds is None:
        west = float(pixel_xy.longitude.min())
        east = float(pixel_xy.longitude.max())
        south = float(pixel_xy.latitude.min())
        north = float(pixel_xy.latitude.max())

        padding_x = max((east - west) * 0.03, 0.02)
        padding_y = max((north - south) * 0.03, 0.02)

        west -= padding_x
        east += padding_x
        south -= padding_y
        north += padding_y

    figure.update_xaxes(
        range=[west, east],
        title_text="Longitude",
        tickformat=".2f",
        showgrid=True,
        gridcolor="#E8EDF3",
        row=row,
        col=col,
    )

    figure.update_yaxes(
        range=[south, north],
        title_text="Latitude",
        tickformat=".2f",
        showgrid=True,
        gridcolor="#E8EDF3",
        scaleanchor=f"x{(row - 1) * 2 + col}",
        scaleratio=1 / np.cos(
            np.deg2rad((south + north) / 2)
        ),
        row=row,
        col=col,
    )


# ------------------------------------------------------------
# Regional extent and zoom bounds
# ------------------------------------------------------------

regional_bounds = [
    float(pixel_xy.longitude.min()),
    float(pixel_xy.latitude.min()),
    float(pixel_xy.longitude.max()),
    float(pixel_xy.latitude.max()),
]

zoom_bounds = {}

for name in ["Tacloban", "Ormoc", "Guiuan"]:
    match = municipalities_display.loc[
        municipalities_display.unit_name.eq(name)
    ]

    if match.empty:
        print(f"{name}: boundary not found; skipping zoom.")
        continue

    west, south, east, north = match.geometry.iloc[0].bounds

    padding_x = max((east - west) * 0.30, 0.025)
    padding_y = max((north - south) * 0.30, 0.025)

    zoom_bounds[name] = [
        west - padding_x,
        south - padding_y,
        east + padding_x,
        north + padding_y,
    ]

# Guiuan mainland; exclude the islands farther south.
zoom_bounds["Guiuan"] = [125.62, 10.97, 125.82, 11.13]

# ------------------------------------------------------------
# Four-panel spatial figure
# ------------------------------------------------------------

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        "All eligible G3 pixels · Samar–Leyte view",
        "Tacloban",
        "Ormoc",
        "Guiuan",
    ],
    horizontal_spacing=0.07,
    vertical_spacing=0.12,
)

# Regional panel
add_family_panel(
    fig,
    row=1,
    col=1,
    title="Samar–Leyte",
    bounds=regional_bounds,
    boundary_names=None,
    show_legend=True,
)

# Zoom panels
for row, col, name in [
    (1, 2, "Tacloban"),
    (2, 1, "Ormoc"),
    (2, 2, "Guiuan"),
]:
    if name not in zoom_bounds:
        continue

    add_family_panel(
        fig,
        row=row,
        col=col,
        title=name,
        bounds=zoom_bounds[name],
        boundary_names=[name],
        show_legend=False,
    )


style_figure(
    fig,
    "Pixel-level recovery families projected to space",
)

fig.update_layout(
    width=1600,
    height=1050,
    margin=dict(
        l=80,
        r=40,
        t=110,
        b=125,
    ),
    legend=dict(
        orientation="h",
        x=0,
        y=-0.08,
        xanchor="left",
        yanchor="top",
        title=dict(text="Pixel family"),
    ),
)

fig.add_annotation(
    x=0,
    y=-0.16,
    xref="paper",
    yref="paper",
    xanchor="left",
    text=(
        "Each point is one eligible G3 pixel. "
        "Colours indicate temporal-family membership; "
        "municipality boundaries provide spatial reference."
    ),
    showarrow=False,
    font=dict(
        family="Arial",
        size=12,
        color="#64748B",
    ),
)

finish_figure(
    fig,
    "pixel_family_spatial_maps",
)

In [ ]:
# ============================================================
# PIXEL FAMILIES, BASELINE NTL, AND GHSL G3
# Samar–Leyte | Tacloban | Ormoc | Guiuan mainland
# ============================================================

# Align family labels with the VIIRS grid.
pixel_xy = pixel_xy.reset_index(drop=True)
assert len(pixel_xy) == len(pixel_family_id)

x_coords = np.asarray(pixel_composites.x.values, dtype=float)
y_coords = np.asarray(pixel_composites.y.values, dtype=float)

x_index = pd.Index(np.round(x_coords, 8)).get_indexer(
    np.round(np.asarray(pixel_xy["x"], dtype=float), 8)
)
y_index = pd.Index(np.round(y_coords, 8)).get_indexer(
    np.round(np.asarray(pixel_xy["y"], dtype=float), 8)
)

family_grid = np.full((len(y_coords), len(x_coords)), np.nan)
located = (x_index >= 0) & (y_index >= 0)
family_grid[y_index[located], x_index[located]] = np.asarray(
    pixel_family_id
)[located]

# Reuse the baseline raster if this cell was already run.
if "baseline_raster" not in globals():
    baseline_raster = (
        rq_cube.sel(date=slice(BASELINE_START, PRE_EVENT_END))
        .median("date", skipna=True)
        .compute()
    )

baseline_x = np.asarray(baseline_raster.x.values, dtype=float)
baseline_y = np.asarray(baseline_raster.y.values, dtype=float)
baseline_grid = np.asarray(baseline_raster.values, dtype=float).copy()
baseline_grid[
    ~np.isfinite(baseline_grid)
    | (baseline_grid < 0)
    | np.isin(baseline_grid, [6553.5, 65535.0, 999999.0])
] = np.nan
baseline_zmax = float(np.nanquantile(baseline_grid, 0.98))

# GHSL is shown as three discrete G3 classes.
ghsl_x = np.asarray(ghsl_viirs.x.values, dtype=float)
ghsl_y = np.asarray(ghsl_viirs.y.values, dtype=float)
ghsl_raw = np.asarray(ghsl_viirs.values)
ghsl_grid = np.full(ghsl_raw.shape, np.nan)
for class_value, colour_index in [(22, 0), (23, 1), (30, 2)]:
    ghsl_grid[ghsl_raw == class_value] = colour_index

def discrete_scale(colours):
    n = len(colours)
    return [
        stop
        for i, colour in enumerate(colours)
        for stop in ([i / n, colour], [(i + 1) / n, colour])
    ]

family_scale = discrete_scale(FAMILY_COLORS[:pixel_selected_k])
ghsl_scale = discrete_scale(["#a87000", "#732600", "#ff0000"])

# Pale land context from the existing municipal boundaries.
regional_context = regional_boundaries.loc[
    regional_boundaries.province_name.isin([
        "Samar", "Eastern Samar", "Northern Samar",
        "Leyte", "Southern Leyte",
    ])
]

west, south, east, north = regional_context.total_bounds
panel_bounds = {
    "Samar–Leyte": [west - 0.05, south - 0.05, east + 0.05, north + 0.05],
}

for name in ["Tacloban", "Ormoc"]:
    geometry = municipalities_display.loc[
        municipalities_display.unit_name.eq(name), "geometry"
    ].iloc[0]
    w, s, e, n = geometry.bounds
    panel_bounds[name] = [
        w - max((e - w) * 0.30, 0.025),
        s - max((n - s) * 0.30, 0.025),
        e + max((e - w) * 0.30, 0.025),
        n + max((n - s) * 0.30, 0.025),
    ]

# Use the mainland town area, not the bounds of Guiuan's islands.
panel_bounds["Guiuan"] = [125.62, 10.97, 125.82, 11.13]

def cropped(grid, xx, yy, bounds):
    w, s, e, n = bounds
    keep_x = np.flatnonzero((xx >= w) & (xx <= e))
    keep_y = np.flatnonzero((yy >= s) & (yy <= n))
    if not len(keep_x) or not len(keep_y):
        return np.empty((0, 0)), np.array([]), np.array([])
    xs = slice(keep_x.min(), keep_x.max() + 1)
    ys = slice(keep_y.min(), keep_y.max() + 1)
    return grid[ys, xs], xx[xs], yy[ys]

def add_land(figure, row, col, bounds):
    # Layout shapes stay below the raster; filled Scatter traces do not.
    clip = box(*bounds)
    paths = []
    for geometry in regional_context.geometry:
        if geometry is None or geometry.is_empty or not geometry.intersects(clip):
            continue
        clipped = geometry.intersection(clip)
        polygons = (
            list(clipped.geoms)
            if clipped.geom_type == "MultiPolygon"
            else [clipped]
        )
        for polygon in polygons:
            if polygon.geom_type != "Polygon" or polygon.is_empty:
                continue
            coords = polygon.exterior.coords
            paths.append(
                "M " + " L ".join(f"{x},{y}" for x, y in coords) + " Z"
            )
    if paths:
        figure.add_shape(
            type="path",
            path=" ".join(paths),
            fillcolor="#E8EDF3",
            line=dict(color="#FFFFFF", width=0.4),
            layer="below",
            row=row,
            col=col,
        )

def add_raster(figure, grid, xx, yy, bounds, row, col,
               colours, zmin, zmax, colourbar=None):
    values, panel_x, panel_y = cropped(grid, xx, yy, bounds)
    if not values.size:
        return
    figure.add_trace(
        go.Heatmap(
            x=panel_x,
            y=panel_y,
            z=values,
            zmin=zmin,
            zmax=zmax,
            colorscale=colours,
            zsmooth=False,
            xgap=0,
            ygap=0,
            hoverongaps=False,
            showscale=colourbar is not None,
            colorbar=colourbar,
        ),
        row=row,
        col=col,
    )

def add_outlines_and_track(figure, row, col, name):
    outlines = (
        municipalities_display
        if name == "Samar–Leyte"
        else municipalities_display.loc[
            municipalities_display.unit_name.eq(name)
        ]
    )
    for geometry in outlines.geometry:
        xx, yy = boundary_coordinates(geometry)
        figure.add_trace(
            go.Scatter(
                x=xx, y=yy, mode="lines",
                line=dict(color="#FFFFFF", width=0.9),
                hoverinfo="skip", showlegend=False,
            ),
            row=row, col=col,
        )

    for geometry in haiyan_track.geometry:
        lines = (
            list(geometry.geoms)
            if geometry.geom_type == "MultiLineString"
            else [geometry]
        )
        for line in lines:
            xx, yy = line.xy
            for colour, width in [("#FFFFFF", 4), ("#111111", 1.8)]:
                figure.add_trace(
                    go.Scatter(
                        x=list(xx), y=list(yy), mode="lines",
                        line=dict(color=colour, width=width),
                        hoverinfo="skip", showlegend=False,
                    ),
                    row=row, col=col,
                )

names = ["Samar–Leyte", "Tacloban", "Ormoc", "Guiuan"]
fig = make_subplots(
    rows=3,
    cols=4,
    subplot_titles=[
        f"{heading} · {name}"
        for heading in [
            "Recovery families",
            "Median baseline NTL",
            "GHSL G3",
        ]
        for name in names
    ],
    horizontal_spacing=0.045,
    vertical_spacing=0.075,
)

style_figure(
    fig,
    "Pixel recovery families, baseline NTL and GHSL G3",
)

for col, name in enumerate(names, start=1):
    bounds = panel_bounds[name]  # Used for cropping AND axis ranges.
    w, s, e, n = bounds

    for row in (1, 2, 3):
        add_land(fig, row, col, bounds)

    add_raster(
        fig, family_grid, x_coords, y_coords, bounds, 1, col,
        family_scale, -0.5, pixel_selected_k - 0.5,
        colourbar=dict(
            title="Family",
            tickvals=list(range(pixel_selected_k)),
            ticktext=[f"F{i + 1}" for i in range(pixel_selected_k)],
            x=1.01, y=0.84, len=0.25, thickness=13,
        ) if col == 1 else None,
    )
    add_raster(
        fig, baseline_grid, baseline_x, baseline_y, bounds, 2, col,
        "Inferno", 0, baseline_zmax,
        colourbar=dict(
            title="Median NTL",
            x=1.01, y=0.50, len=0.25, thickness=13,
        ) if col == 1 else None,
    )
    add_raster(
        fig, ghsl_grid, ghsl_x, ghsl_y, bounds, 3, col,
        ghsl_scale, -0.5, 2.5,
        colourbar=dict(
            title="GHSL G3",
            tickvals=[0, 1, 2],
            ticktext=["22", "23", "30"],
            x=1.01, y=0.16, len=0.25, thickness=13,
        ) if col == 1 else None,
    )

    for row in (1, 2, 3):
        add_outlines_and_track(fig, row, col, name)
        axis_number = (row - 1) * 4 + col
        fig.update_xaxes(
            range=[w, e],
            title_text="Longitude" if row == 3 else None,
            tickformat=".2f",
            gridcolor="#E5EBF3",
            row=row, col=col,
        )
        fig.update_yaxes(
            range=[s, n],
            title_text="Latitude" if col == 1 else None,
            tickformat=".2f",
            gridcolor="#E5EBF3",
            scaleanchor=f"x{axis_number}",
            scaleratio=1 / np.cos(np.deg2rad((s + n) / 2)),
            row=row, col=col,
        )

# Study-area labels on the regional panels.
for area in municipalities_display.itertuples():
    if area.unit_name == "Guiuan":
        label_x, label_y = 125.7237, 11.0307
    else:
        point = area.geometry.representative_point()
        label_x, label_y = point.x, point.y

    dx, dy = map_offsets[area.unit_name]
    for row in (1, 2, 3):
        fig.add_annotation(
            x=label_x, y=label_y,
            text=area.unit_name,
            ax=dx, ay=dy,
            showarrow=True,
            arrowhead=0,
            arrowwidth=0.7,
            arrowcolor="#64748B",
            font=dict(family="Arial", size=12, color="#243B5A"),
            bgcolor="rgba(255,255,255,0.82)",
            borderpad=1,
            row=row, col=1,
        )

fig.update_layout(
    width=1600,
    height=1200,
    margin=dict(l=75, r=130, t=110, b=60),
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="#FFFFFF",
    font=dict(family="Arial", size=14, color="#243B5A"),
)
fig.update_annotations(font=dict(family="Arial", size=11, color="#243B5A"))

guiuan_family, _, _ = cropped(
    family_grid, x_coords, y_coords, panel_bounds["Guiuan"]
)
print(
    "Guiuan mainland classified pixels:",
    int(np.isfinite(guiuan_family).sum()),
)

finish_figure(fig, "pixel_family_baseline_ghsl_grid_maps")

## Export and next step

Exports retain the original profile, anchor, feature, family and baseline-diagnostic filenames where their meaning is unchanged. Additional tables retain both supports, bin counts, excluded trajectories and k-selection diagnostics. The settings file records the signal, baseline, spatial threshold, GHSL class and event window. No missing observations are filled, and excluded locations remain visible.

**Next stage:** compare the observed families and their quality flags with physical and life-quality layers. That comparison is intentionally outside this notebook.


In [ ]:
TABLE_DIR.mkdir(parents=True, exist_ok=True)
municipality_four_day.to_csv(TABLE_DIR / "focused_municipality_four_day_profiles.csv", index=False)
kernel_four_day.to_csv(TABLE_DIR / "focused_5x5_four_day_profiles.csv", index=False)
kernel_anchors.to_csv(TABLE_DIR / "focused_5x5_anchors.csv", index=False)
complete_features.reset_index().to_csv(TABLE_DIR / "focused_clustering_features.csv", index=False)
family_id.rename("family_id").reset_index().to_csv(TABLE_DIR / "focused_recovery_families.csv", index=False)
brightness_diagnostic.to_csv(TABLE_DIR / "focused_baseline_variability.csv", index=False)
clustering_features.to_csv(TABLE_DIR / "focused_all_trajectory_features.csv")
clustering_counts.to_csv(TABLE_DIR / "focused_all_trajectory_counts.csv")
complete_features_all.to_csv(TABLE_DIR / "focused_clustering_features_both_supports.csv")
eligibility.reset_index().to_csv(TABLE_DIR / "focused_clustering_eligibility.csv", index=False)
diagnostics_all.to_csv(TABLE_DIR / "focused_k_diagnostics.csv", index=False)
assignments.to_csv(TABLE_DIR / "focused_recovery_families_both_supports.csv", index=False)
family_summaries.to_csv(TABLE_DIR / "focused_family_summaries.csv", index=False)
settings = dict(signal=DNB_BAND, mqf=0, ghsl_mask=SETTLEMENT_MASK, ghsl_classes=GHSL_MASKS[SETTLEMENT_MASK],
    event_date=str(EVENT_DATE.date()), baseline_start=str(BASELINE_START.date()), baseline_end=str(PRE_EVENT_END.date()),
    analysis_start=str(ANALYSIS_START.date()), profile_end=str(PROFILE_END.date()),
    baseline_pixel_min_composites=MIN_BASELINE_OBSERVATIONS[4], spatial_completeness_pct=SPATIAL_COMPLETENESS_PCT,
    aggregation_days=AGGREGATION_DAYS, daily_cap_percentile=RQ_CLIP_PERCENTILE,
    cluster_horizon_days=CLUSTER_HORIZON_DAYS, cluster_bin_days=CLUSTER_BIN_DAYS,
    minimum_bin_composites=MIN_BIN_COMPOSITES, imputation="none", scaling="none: baseline-relative percentage",
    random_state=CLUSTER_RANDOM_STATE, clustering="separate by spatial support", study_areas=STUDY_AREAS)
(TABLE_DIR / "focused_analysis_settings.json").write_text(json.dumps(settings, indent=2))
print("Tables:", TABLE_DIR)
print("Interactive figures:", FIGURE_DIR)


Method reference: [scikit-learn silhouette analysis](https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.html). Silhouettes are defined only for 2 through n−1 realised labels. A selected k is a candidate for this small observed subset, not a demonstrated population optimum.

**Execution note.** The notebook structure and Python syntax were checked during refurbishment. The local source rasters and boundary attributes were inaccessible; full execution and empirical k-selection remain to be performed in the project environment. The old T50/T80/T90 module has been omitted from this minimum version to keep the requested trajectory-to-family sequence concise; this notebook does not overwrite those older metric files.
